# Retail Sales — Exploratory Data Analysis

## 1. Business Context & Objectives
This project explores retail transactions to identify sales trends, customer behaviour, category performance, and actionable business insights.

**Tools:** Python 3.11.15, pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

**Project:** OASIS Infobyte Data Analytics — Level 1, Task 1.

## 2. Business Questions
1. How does revenue evolve over time?
2. Are there monthly or quarterly sales patterns?
3. Which categories generate the most revenue?
4. How does purchasing behaviour differ by gender?
5. How does transaction value vary across age groups?
6. Which customers contribute the most revenue?
7. How do categories differ in volume, revenue, average transaction value and profitability?
8. What relationships exist among age, quantity, price, COGS and sales?
9. Are there unusual transactions or outliers?
10. What additional pattern can support business decisions?

**Data limitation:** the dataset has `category` but no product identifier/name, so a true Top-10-products analysis is not possible. Category-level analysis is used instead.

## 3. Environment & Libraries

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python version: {sys.version.split()[0]}")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")

## 4. Data Loading

In [ ]:
DATA_PATH = "../data/Retail_Sales.csv"
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

In [ ]:
df_raw.info()
df_raw.describe().T

## 5. Data Understanding & Quality Assessment

In [ ]:
df_raw.nunique().sort_values()

In [ ]:
print("Gender values:")
print(df_raw["gender"].value_counts(dropna=False))
print("\nCategory values:")
print(df_raw["category"].value_counts(dropna=False))
print("\nUnique transactions:", df_raw["transactions_id"].nunique())
print("Unique customers:", df_raw["customer_id"].nunique())

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percentage": df_raw.isna().mean() * 100
}).sort_values("missing_count", ascending=False)
missing_report

In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())
raw_numeric_cols = ["age", "quantiy", "price_per_unit", "cogs", "total_sale"]
for col in raw_numeric_cols:
    print(f"{col}: {(df_raw[col] <= 0).sum()} non-positive values")

## 6. Data Preparation & Validation

In [ ]:
# Preserve the raw dataset; perform transformations on a separate working copy.
df_clean = df_raw.copy()

# Correct the source typo and convert the transaction date.
df_clean = df_clean.rename(columns={"quantiy": "quantity"})
df_clean["sale_date"] = pd.to_datetime(df_clean["sale_date"], errors="coerce")
df_clean.dtypes

### 6.1 Validate Total Sales

For complete records, **total sale = quantity × price per unit**. This validation also tells us whether missing sales can be reconstructed directly.

In [ ]:
df_clean["calculated_total_sale"] = df_clean["quantity"] * df_clean["price_per_unit"]
df_clean["sales_difference"] = df_clean["total_sale"] - df_clean["calculated_total_sale"]
print("Inconsistent complete transactions:", df_clean["sales_difference"].abs().gt(0.01).sum())

reconstructable = df_clean["total_sale"].isna() & df_clean["quantity"].notna() & df_clean["price_per_unit"].notna()
df_clean.loc[reconstructable, "total_sale"] = df_clean.loc[reconstructable, "quantity"] * df_clean.loc[reconstructable, "price_per_unit"]
print("Reconstructed total_sale values:", reconstructable.sum())
print("Remaining missing total_sale values:", df_clean["total_sale"].isna().sum())

### 6.2 Missing-Value Treatment

Missing values are not blindly replaced with global medians. `total_sale` is reconstructed when quantity and unit price provide a direct calculation. Other missing fields are retained where reliable reconstruction is not justified.

In [ ]:
df_clean[df_clean.isna().any(axis=1)]

In [ ]:
df_clean["year"] = df_clean["sale_date"].dt.year
df_clean["month"] = df_clean["sale_date"].dt.month
df_clean["quarter"] = df_clean["sale_date"].dt.to_period("Q").astype(str)
df_clean["profit"] = df_clean["total_sale"] - df_clean["cogs"]

bins = [0, 24, 34, 44, 54, 64, np.inf]
labels = ["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"]
df_clean["age_group"] = pd.cut(df_clean["age"], bins=bins, labels=labels)
df_clean.head()

# 7. Exploratory Data Analysis

Each section combines quantitative summaries with visual evidence. Interpretations should be based on the executed outputs.

## 7.1 Descriptive Statistics

In [ ]:
numeric_cols = ["age", "quantity", "price_per_unit", "cogs", "total_sale", "profit"]
descriptive_stats = pd.DataFrame({
    "mean": df_clean[numeric_cols].mean(),
    "median": df_clean[numeric_cols].median(),
    "mode": df_clean[numeric_cols].mode().iloc[0],
    "std": df_clean[numeric_cols].std()
})
descriptive_stats

**Observation:** Compare mean and median to identify skewness and use standard deviation to assess variability. Large gaps may indicate skewed distributions or influential observations.

## 7.2 Monthly and Quarterly Sales Trends

In [ ]:
monthly_sales = (df_clean.dropna(subset=["sale_date", "total_sale"]).set_index("sale_date").resample("MS").agg(
    revenue=("total_sale", "sum"), transactions=("transactions_id", "count"), quantity_sold=("quantity", "sum"), profit=("profit", "sum")
) .reset_index())
monthly_sales["average_transaction_value"] = monthly_sales["revenue"] / monthly_sales["transactions"]
monthly_sales.head()

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_sales, x="sale_date", y="revenue", marker="o")
plt.title("Monthly Retail Sales Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Observation:** The monthly trend reveals periods of stronger and weaker revenue. Peaks should be interpreted alongside transaction volume and average transaction value.

In [ ]:
quarterly_sales = (df_clean.dropna(subset=["sale_date", "total_sale"]).groupby("quarter").agg(
    revenue=("total_sale", "sum"), transactions=("transactions_id", "count"), quantity_sold=("quantity", "sum"), profit=("profit", "sum")
) .reset_index())
quarterly_sales

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=quarterly_sales, x="quarter", y="revenue", marker="o")
plt.title("Quarterly Retail Sales Revenue")
plt.xlabel("Quarter")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Observation:** Quarterly aggregation highlights broader seasonal movement. Strong quarters should be investigated using category and customer metrics.

## 7.3 Product Category Performance

The dataset has no individual product identifier, so category-level analysis is the most detailed valid product analysis available.

In [ ]:
category_summary = df_clean.groupby("category").agg(
    revenue=("total_sale", "sum"), transactions=("transactions_id", "count"), quantity_sold=("quantity", "sum"), average_transaction=("total_sale", "mean"), profit=("profit", "sum")
).sort_values("revenue", ascending=False)
category_summary["profit_margin_pct"] = category_summary["profit"] / category_summary["revenue"] * 100
category_summary

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=category_summary.reset_index(), x="category", y="revenue")
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

**Observation:** Revenue ranking identifies the strongest categories by sales value. Revenue leadership does not automatically mean profitability leadership.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.barplot(data=category_summary.reset_index(), x="category", y="transactions", ax=axes[0])
axes[0].set_title("Transactions by Category")
sns.barplot(data=category_summary.reset_index(), x="category", y="average_transaction", ax=axes[1])
axes[1].set_title("Average Transaction Value")
sns.barplot(data=category_summary.reset_index(), x="category", y="profit_margin_pct", ax=axes[2])
axes[2].set_title("Profit Margin by Category")
plt.tight_layout()
plt.show()

**Observation:** Comparing volume, average transaction value and margin distinguishes categories that sell frequently from categories that generate higher value or profitability per transaction.

## 7.4 Customer Demographics — Gender

In [ ]:
gender_summary = df_clean.groupby("gender").agg(
    transactions=("transactions_id", "count"), revenue=("total_sale", "sum"), average_transaction=("total_sale", "mean")
)
gender_summary["revenue_share_pct"] = gender_summary["revenue"] / gender_summary["revenue"].sum() * 100
gender_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=gender_summary.reset_index(), x="gender", y="revenue", ax=axes[0])
axes[0].set_title("Revenue by Gender")
sns.barplot(data=gender_summary.reset_index(), x="gender", y="average_transaction", ax=axes[1])
axes[1].set_title("Average Transaction Value by Gender")
plt.tight_layout()
plt.show()

**Observation:** Gender comparisons use both total revenue and average transaction value because total revenue is affected by transaction volume.

## 7.5 Customer Demographics — Age Groups

In [ ]:
age_summary = df_clean.dropna(subset=["age_group"]).groupby("age_group", observed=True).agg(
    customers=("customer_id", "nunique"), transactions=("transactions_id", "count"), revenue=("total_sale", "sum"), average_transaction=("total_sale", "mean")
) .reset_index()
age_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=age_summary, x="age_group", y="transactions", ax=axes[0])
axes[0].set_title("Transactions by Age Group")
sns.barplot(data=age_summary, x="age_group", y="average_transaction", ax=axes[1])
axes[1].set_title("Average Transaction Value by Age Group")
plt.tight_layout()
plt.show()

**Observation:** Age-group analysis separates customer volume from spending behaviour. Differences in average transaction value may support targeted marketing tests, but do not establish causation.

## 7.6 Customer Revenue Concentration

In [ ]:
customer_summary = df_clean.groupby("customer_id").agg(
    transactions=("transactions_id", "count"), revenue=("total_sale", "sum"), average_transaction=("total_sale", "mean")
) .sort_values("revenue", ascending=False)
customer_summary["cumulative_revenue_share_pct"] = customer_summary["revenue"].cumsum() / customer_summary["revenue"].sum() * 100
top20_n = max(1, int(np.ceil(len(customer_summary) * 0.20)))
top20_share = customer_summary.head(top20_n)["revenue"].sum() / customer_summary["revenue"].sum() * 100
print(f"Customers analysed: {len(customer_summary):,}")
print(f"Top 20% revenue share: {top20_share:.2f}%")

In [ ]:
plt.figure(figsize=(10, 6))
rank = np.arange(1, len(customer_summary) + 1)
plt.plot(rank, customer_summary["cumulative_revenue_share_pct"])
plt.axvline(top20_n, linestyle="--", label=f"Top 20% ({top20_n:,} customers)")
plt.axhline(80, linestyle="--", label="80% revenue")
plt.title("Cumulative Customer Revenue Contribution")
plt.xlabel("Customers ranked by revenue")
plt.ylabel("Cumulative revenue share (%)")
plt.legend()
plt.tight_layout()
plt.show()

**Observation:** The actual top-20% revenue share is measured from the data; an 80/20 relationship is not assumed.

## 7.7 Correlation Analysis

Correlation measures linear association, not causation. Relationships involving `total_sale`, `quantity`, and `price_per_unit` require caution because total sales is mathematically derived from quantity and unit price.

In [ ]:
correlation_cols = ["age", "quantity", "price_per_unit", "cogs", "total_sale", "profit"]
correlation_matrix = df_clean[correlation_cols].corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

**Observation:** The heatmap identifies the direction and strength of linear relationships. Mathematically linked variables should not be presented as causal findings.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_clean, x="quantity", y="total_sale", alpha=0.5, ax=axes[0])
axes[0].set_title("Quantity vs. Total Sale")
sns.scatterplot(data=df_clean, x="price_per_unit", y="total_sale", alpha=0.5, ax=axes[1])
axes[1].set_title("Unit Price vs. Total Sale")
plt.tight_layout()
plt.show()

**Observation:** The scatter plots visually assess the main relationships and can reveal unusual observations.

## 7.8 Additional Insight — What Drives High-Revenue Months?

In [ ]:
monthly_sales.sort_values("revenue", ascending=False).head(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
sns.lineplot(data=monthly_sales, x="sale_date", y="transactions", marker="o", ax=axes[0])
axes[0].set_title("Monthly Transaction Volume")
sns.lineplot(data=monthly_sales, x="sale_date", y="average_transaction_value", marker="o", ax=axes[1])
axes[1].set_title("Monthly Average Transaction Value")
plt.tight_layout()
plt.show()

**Observation:** Comparing transaction count and average transaction value helps determine whether revenue peaks are driven mainly by more purchases, larger purchases, or both.

# 8. Key Findings

After executing the notebook, summarise at least three findings with actual values. Avoid unsupported claims and distinguish observations from interpretation.

# 9. Business Recommendations

Recommendations should follow from the validated findings:

1. **Seasonal planning:** Align inventory, staffing and promotions with periods of consistently higher demand.
2. **Category strategy:** Prioritise categories combining strong revenue and healthy margins; review weaker categories for pricing, assortment or cost efficiency.
3. **Customer retention:** If revenue is concentrated among a small group, develop targeted retention initiatives for high-value customers.
4. **Segmented marketing:** Use observed age/gender differences to test targeted campaigns.
5. **Improve data collection:** Add product identifiers and richer customer attributes to enable product rankings and deeper segmentation.

# 10. Limitations & Conclusion

## Limitations
- No individual product identifier/name is available, so a Top-10-products ranking cannot be produced.
- Customer information is limited mainly to age and gender.
- Correlation does not imply causation.
- Missing values remain where reliable reconstruction is not possible.
- The dataset covers two calendar years, limiting long-term seasonal inference.

## Conclusion
This analysis evaluates retail sales trends, category performance, customer demographics, customer concentration and numerical relationships. The final conclusion should be based on the executed outputs and should clearly distinguish observed patterns, interpretation and recommendations.